In [ ]:
"""
对批量标准化进行标准化/对神经网络进行标准化 提高批次的数量
调整优化器的学习率
优化器选择用adam而不是sgd
增加神经网络的深度
使用随机失活 避免过拟合
"""

In [ ]:
# 导包
import torch
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset,DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torchsummary import summary
import time

In [19]:
def create_dataset():
    # 构建数据集
    data = pd.read_csv("手机价格预测.csv",encoding='gbk')
    # print(data)
    # 划分特征值和标签
    x, y = data.iloc[:, :-1], data.iloc[:, -1]
    x = x.astype(np.float32)
    # 标签的长度都是int64 可能有的标签特别长
    y = y.astype(np.int64)
    # print(f"x:{x}")
    # print(f"y:{y}")
    # 划分训练集和测试集
    x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)
    train_dataset = TensorDataset(torch.from_numpy(x_train.values),torch.from_numpy(y_train.values))
    test_dataset = TensorDataset(torch.from_numpy(x_test.values),torch.from_numpy(y_test.values))
    # 返回结果
    return train_dataset,test_dataset,x_train.shape[1],len(np.unique(y))

In [ ]:
# 构建网络模型
class Phone_Price(nn.Module):
    def __init__(self, input_dim, out_dim):
        # 调用父类方法
        super(Phone_Price, self).__init__()
        # 构建第一个神经层 同时对神经层进行标准化
        self.linear1 = nn.Linear(input_dim,128)
        self.ln1 = nn.BatchNorm1d(128)
        nn.init.kaiming_normal_(self.linear1.weight,nonlinearity='relu')
        nn.init.zeros_(self.linear1.bias)
        dropout1 = nn.Dropout(0.4)
        # 构建第二个神经层 同时对神经层进行标准化
        self.linear2 = nn.Linear(128,256)
        self.ln2 = nn.BatchNorm1d(256)
        nn.init.kaiming_normal_(self.linear2.weight,nonlinearity='relu')
        nn.init.zeros_(self.linear2.bias)
        dropout2 = nn.Dropout(0.4)
        # 构建第三个神经层
        self.linear3 = nn.Linear(256,out_dim)
    def forward(self,x):
        # 向前传播 同时使用relu激活
        x = torch.relu(self.ln1(self.linear1(x)))
        x = torch.relu(self.ln2(self.linear2(x)))
        x = dropout1(x)
        x = dropout2(x)
        # 输出
        output = self.linear3(x)
        return output



In [21]:
def model_train(train_dataset, input_dim, out_dim):
    # 模型训练
    # 创建数据加载器 用于批量处理大数据
    dataloader = DataLoader(train_dataset, batch_size=15, shuffle=True)
    # 创建模型
    model = Phone_Price(input_dim, out_dim)
    # 创建损失函数 使用交叉熵函数
    loss_fn = nn.CrossEntropyLoss()
    # 创建优化器
    optimizer = optim.Adam(model.parameters(),lr=0.01)
    # 定义训练轮次
    epochs = 300
    # 训练模型
    for epoch in range(epochs):
        # 定义总样本数量 总损失值
        batch_num = 0
        total_loss = 0.0
        start = time.time()
        # 对每一轮的数据集进行训练
        for x,y in dataloader:
            # 构建模型
            model.train()
            # 向前传播
            y_pred = model(x)
            # 梯度清零
            optimizer.zero_grad()
            # 反向传播
            loss = loss_fn(y_pred,y)
            loss.backward()
            # 更新权重
            optimizer.step()
            # 记住训练信息
            batch_num += 1
            total_loss += loss.item()
        # 一个轮次训练完成
        print(f"epoch:{epoch+1},Loss:{total_loss/batch_num},time:{time.time()-start}")
    print(f"模型参数:{model.state_dict()}")
    # 保存模型
    torch.save(model.state_dict(),"phone_price.pkl")

In [22]:
# 模型评估
def model_eva(test_dataset, input_dim, out_dim):
    # 创建数据加载器
    dataloader = DataLoader(test_dataset, batch_size=15)
    # 创建模型对象
    model = Phone_Price(input_dim, out_dim)
    # 加载模型
    model.load_state_dict(torch.load('phone_price.pkl'))
    # 定义预测正确的数量
    corrent_num = 0
    # 按批次运行和进行评估
    for x,y in dataloader:
        # 修改模型状态为预测模型
        model.eval()
        # 模型预测
        y_pred = model(x)
        # softmax激活 将模型的评分转换为概率
        y_pred = torch.softmax(y_pred,dim=-1)
        #返回概率最大类别索引
        y_pred = torch.argmax(y_pred, dim=-1)
        # 计算正确的数量
        corrent_num += (y_pred == y).sum()
    # 计算完毕 打印正确的结果
    print(f"测试集的规模：{len(test_dataset)}")
    print(f"正确数量：{corrent_num}")
    print(f"正确率：{corrent_num/len(test_dataset) * 100}")   

In [34]:
if __name__ == '__main__':
    train_dataset,test_dataset,input_dim,out_dim = create_dataset()
    print(f"输入的特征数:{input_dim}")
    print(f"输出的类别数:{out_dim}")
    # 神经网络对象创建
    # 创建模型
    model = Phone_Price(input_dim, out_dim)
    summary(model, input_size=(input_dim,),batch_size=5)
    # 模型训练
    model_train(train_dataset, input_dim, out_dim)
    # 模型评估
    model_eva(test_dataset, input_dim, out_dim)
  
    

输入的特征数:20
输出的类别数:4
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [5, 128]           2,688
       BatchNorm1d-2                   [5, 128]             256
            Linear-3                   [5, 256]          33,024
       BatchNorm1d-4                   [5, 256]             512
            Linear-5                     [5, 4]           1,028
Total params: 37,508
Trainable params: 37,508
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.03
Params size (MB): 0.14
Estimated Total Size (MB): 0.17
----------------------------------------------------------------
epoch:1,Loss:0.6633386781839566,time:0.24785661697387695
epoch:2,Loss:0.5670058521154885,time:0.3708767890930176
epoch:3,Loss:0.5320057745709598,time:0.6554739475250244
epoch:4,Loss:0.5693739988257952,time:0.630192518